# Notebook 06 — Reproducing Research Tables and Figures

**Purpose:** Link interactive notebook exploration directly to DERMA-ACT V2's standalone CLI reproducibility pipeline, execute safe aggregate table (`13` LaTeX tables) and aggregate figure (`Figure 1`, `Figure 4`, `Graphical Abstract`) generation into an isolated ignored output directory, verify SHA-256 parity against tracked reference tables, and map every manuscript table/figure to its exact artifact requirements.  
**Data & Artifact Requirement:** **Standalone Clean-Room Capable (Aggregate Tier).** Never retrains models and never modifies frozen manuscript files or tracked source directories. Full Figures 2, 3, 5, 6, 7, and 8 require local ISIC imagery and frozen `.npz` caches (`REQUIRES_DATA` / `REQUIRES_MODEL_ARTIFACT`).

## 1. Learning Objectives & Scientific Context

### Learning Objectives
1. Verify environment readiness and inspect all standalone vs. data-dependent reproduction commands.
2. Execute `scripts/generate_tables.py` cleanly into `outputs/notebooks/nb06_reproduction/tables/` and verify exact SHA-256 parity against tracked `supplementary/tables/*.tex`.
3. Execute `scripts/build_figures.py --tier aggregate` cleanly into `outputs/notebooks/nb06_reproduction/figures/` and inspect generated artifacts.
4. Review the complete provenance map of manuscript Tables 1–10 (+ S1, S2, S5) and Figures 1–8 without modifying any frozen source files.

### Prerequisites
- Core packages (`requirements.txt`).

## 2. Environment & Reproduction Script Preflight

In [ ]:
from __future__ import annotations

import json
import os
import platform
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt


def find_repo_root(start: Path | None = None) -> Path:
    """Locate repository root portably without hardcoded machine paths."""
    cur = (start or Path.cwd()).resolve()
    for candidate in [cur, *cur.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "research" / "derma_act" / "config.py").is_file():
            return candidate
    raise FileNotFoundError("Could not locate DERMA-ACT repository root containing pyproject.toml and research/derma_act/config.py")


def find_tables_dir(repo_root: Path) -> Path | None:
    for rel in [
        Path("supplementary") / "tables",
        Path("manuscripts") / "current" / "PAPER_DRAFT_Q2_V3_2_1_PRE_SUBMISSION" / "tables",
    ]:
        cand = repo_root / rel
        if cand.is_dir():
            return cand
    return None


SEED = 42
random.seed(SEED)
np.random.seed(SEED)

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import hashlib
import subprocess

OUT_BASE = REPO_ROOT / "outputs" / "notebooks" / "nb06_reproduction"
OUT_TABLES = OUT_BASE / "tables"
OUT_FIGS = OUT_BASE / "figures"
OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_FIGS.mkdir(parents=True, exist_ok=True)

GEN_TABLES_SCRIPT = REPO_ROOT / "scripts" / "generate_tables.py"
BUILD_FIGS_SCRIPT = REPO_ROOT / "scripts" / "build_figures.py"
REF_TABLES_DIR = find_tables_dir(REPO_ROOT)

display(pd.DataFrame([
    {"script_or_dir": "scripts/generate_tables.py", "available": GEN_TABLES_SCRIPT.is_file(), "tier": "VERIFIED (Aggregate)"},
    {"script_or_dir": "scripts/build_figures.py", "available": BUILD_FIGS_SCRIPT.is_file(), "tier": "VERIFIED (--tier aggregate) / REQUIRES_DATA (--tier all)"},
    {"script_or_dir": "Reference tables directory", "available": REF_TABLES_DIR is not None, "tier": REF_TABLES_DIR.relative_to(REPO_ROOT).as_posix() if REF_TABLES_DIR else "NONE"},
]))

## 3. Artifact Provenance Matrix (Tables & Figures)

| Deliverable | Script | Reproducibility Tier | Required Inputs |
| :--- | :--- | :---: | :--- |
| Tables 1–10, S1, S2, S5 (`13` `.tex` files) | `scripts/generate_tables.py` | `VERIFIED` | Tracked aggregate CSV/JSON ledgers under `research/derma_act/` & `data/aggregate/` |
| Figure 1 (Architecture), Figure 4 (Risk-Cost), Graphical Abstract | `scripts/build_figures.py --tier aggregate` | `VERIFIED` | Tracked aggregate CSV/JSON ledgers only |
| Figures 2, 3, 5, 6, 7, 8 | `scripts/build_figures.py --tier all --root <DATA_ROOT>` | `REQUIRES_DATA` / `REQUIRES_MODEL_ARTIFACT` | Raw ISIC 2019 `.jpg` imagery (`ISIC_0012970.jpg`, etc.) and frozen `.npz` logits / trajectories |

## 4. Regenerate All 13 Aggregate LaTeX Tables (`VERIFIED`)

In [ ]:
proc_tab = subprocess.run(
    [sys.executable, str(GEN_TABLES_SCRIPT), "--root", str(REPO_ROOT), "--out-dir", str(OUT_TABLES)],
    cwd=REPO_ROOT,
    text=True,
    capture_output=True,
    check=True,
)
print(proc_tab.stdout.strip())

gen_files = sorted(OUT_TABLES.glob("*.tex"))
assert len(gen_files) == 13, f"Expected 13 generated .tex tables, got {len(gen_files)}"

rows = []
for gf in gen_files:
    sha_gen = hashlib.sha256(gf.read_bytes()).hexdigest()
    ref_p = (REF_TABLES_DIR / gf.name) if REF_TABLES_DIR else None
    sha_match = (hashlib.sha256(ref_p.read_bytes()).hexdigest() == sha_gen) if (ref_p and ref_p.is_file()) else None
    rows.append({
        "table_file": gf.name,
        "bytes": gf.stat().st_size,
        "sha256_prefix": sha_gen[:16],
        "exact_match_vs_tracked_ref": sha_match,
    })

tables_audit_df = pd.DataFrame(rows)
print(f"Generated {len(tables_audit_df)} tables cleanly in {OUT_TABLES.relative_to(REPO_ROOT)}")
display(tables_audit_df)

## 5. Regenerate Standalone Aggregate Figures (`VERIFIED`) & Inspect Output

In [ ]:
import matplotlib.image as mpimg

proc_fig = subprocess.run(
    [sys.executable, str(BUILD_FIGS_SCRIPT), "--tier", "aggregate", "--out-dir", str(OUT_FIGS)],
    cwd=REPO_ROOT,
    text=True,
    capture_output=True,
    check=True,
)
print(proc_fig.stdout.strip())

# Collect generated aggregate PNG outputs
fig_pngs = sorted(OUT_FIGS.rglob("*.png"))
ga_dir = REPO_ROOT / "outputs" / "graphical_abstract"
if ga_dir.is_dir():
    fig_pngs.extend(sorted(ga_dir.glob("*.png")))

fig_rows = [
    {"output_file": p.relative_to(REPO_ROOT).as_posix(), "bytes": p.stat().st_size, "sha256_prefix": hashlib.sha256(p.read_bytes()).hexdigest()[:16]}
    for p in fig_pngs
]
display(pd.DataFrame(fig_rows))

# Visual preview of regenerated Figure 4 from disk
fig4_png = OUT_FIGS / "fig4_multiview_risk.png"
if fig4_png.is_file():
    img = mpimg.imread(str(fig4_png))
    fig, ax = plt.subplots(figsize=(10, 5.2))
    ax.imshow(img)
    ax.axis("off")
    ax.set_title("Regenerated Aggregate Figure 4 Preview (scripts/build_figures.py --tier aggregate)")
    plt.tight_layout()
    plt.show()
else:
    print("Note: fig4_multiview_risk.png not found at expected path:", fig4_png)

## 6. Interpretation, Verification Boundaries & References

- **Bit-Identical Table Provenance:** Every generated `.tex` table derives solely from tracked Tier A aggregate ledgers under `research/derma_act/` and `data/aggregate/`, leaving frozen manuscript directories and source artifacts untouched.
- **Boundary for `--tier all`:** Regenerating Figures 2, 3, 5, 6, 7, and 8 from raw images and per-image `.npz` logits requires local `CC-BY-NC 4.0` ISIC 2019 data and excluded binary artifacts (`REQUIRES_DATA` / `REQUIRES_MODEL_ARTIFACT`).
- **No Silent Model Retraining:** This workflow never executes end-to-end training or alters checkpoint weights.

### Equivalent CLI Commands
```bash
python scripts/generate_tables.py --out-dir ./outputs/tables                        # [VERIFIED]
python scripts/build_figures.py --tier aggregate --out-dir ./outputs/figures        # [VERIFIED]
python scripts/build_figures.py --tier all --root <DATA_ROOT> --out-dir ./outputs   # [REQUIRES_DATA]
```

### References
- [Reproducibility Guide](../provenance/REPRODUCIBILITY.md) · [Scripts Index](../scripts/README.md) · [Artifacts Registry](../artifacts/README.md) · [Root README](../README.md)